In [1]:
import os
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from dotenv import load_dotenv
import shutil

# 1. 加载环境变量
load_dotenv()

# 2. 配置嵌入模型（硅基流动）
embeddings = OpenAIEmbeddings(
    model="BAAI/bge-large-zh-v1.5",
    openai_api_key=os.getenv("siliconFlow_API_KEY"),
    openai_api_base="https://api.siliconflow.cn/v1"
)

# 3. 加载文档
loader = TextLoader("./data/library.txt", encoding="utf-8")
documents = loader.load()
print(f"加载了 {len(documents)} 个文档")

# 4. 切分文档
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300, chunk_overlap=30,
    separators=["\n\n", "\n", "。", "！", "？", "，", " ", ""]
)
chunks = text_splitter.split_documents(documents)
print(f"文档被切分为 {len(chunks)} 个块")





加载了 1 个文档
文档被切分为 10 个块


In [2]:
import gc

# 删除 vectorstore 变量（如果存在）
if 'vectorstore' in locals():
    del vectorstore
    gc.collect()  # 强制回收内存
    
REBUILD = 1

if REBUILD and os.path.exists("./chroma_db"):
    shutil.rmtree("./chroma_db")
    print("已删除旧数据库，将重新构建。")

已删除旧数据库，将重新构建。


In [2]:
# 5. 创建向量数据库（持久化）
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)
print("向量数据库创建完成！")

# 6. 设置用户问题
user_question = "方闻馆详细时间"  # 你可以修改

# 7. 检索相似块（相似性检索）
retrieved_docs = vectorstore.similarity_search(user_question, k=5)
print(f"检索到的相关块（共 {len(retrieved_docs)} 个）：")
for i, doc in enumerate(retrieved_docs):
    print(f"块{i+1}: {doc.page_content[:150]}...")

# 8. 配置聊天模型
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="deepseek-ai/DeepSeek-V3",
    openai_api_key=os.getenv("siliconFlow_API_KEY"),
    openai_api_base="https://api.siliconflow.cn/v1",
    temperature=0.3
)

# 9. 定义 RAG 问答函数
def rag_answer(question, vectorstore, k=3):
    docs = vectorstore.similarity_search(question, k=k)
    context = "\n\n".join([doc.page_content for doc in docs])
    prompt = f"""你是一个校园助手。请根据以下参考资料回答用户的问题。
如果参考资料中没有相关信息，请如实说“我不知道”。

参考资料：
{context}

问题：{question}
回答："""
    response = llm.invoke(prompt)
    return response.content

# 10. 测试问答
answer = rag_answer(user_question, vectorstore)
print("\n最终答案：", answer)

向量数据库创建完成！
检索到的相关块（共 5 个）：
块1: 一、借阅权限
读者类型              外借册数    预约册数    预约催还过期停借册数
─────────────────────────────────────────────────────────────
在校教职工            100册       30册     ...
块2: 企编教工              30册        3册         1册
访问学者              30册        3册         1册
离退休人员            5册         3册         1册
校内其他人员          5册    ...
块3: 浙江大学图书馆使用指南
═══════════════════════════════════════════════════════

【图书馆概况】
浙江大学图书馆是浙江省规模最大、设备最齐全的高等学校图书馆之一，由紫金港校区的主馆、基础馆、农医馆、古籍馆、方闻馆和玉泉分馆、西溪分馆、华家池分馆...
块4: 【联系方式】
─────────────────────────────────
馆长信箱：infolib@zju.edu.cn
留言板：http://libweb.zju.edu.cn/55982/list.htm
海宁国际校区图书馆：0571-87572288

════════════════...
块5: 三、方闻馆详细时间
─────────────────────────────────
咨询台：周一至周日 8:30-22:30
东阅览室：周一至周日 8:30-22:30
西阅览室：周一至周日 8:30-22:30
珍本阅览室：周一至周五 08:30-12:00；13:30-17:30（周六日不开...

最终答案： 我不知道。


In [3]:
# 5. 创建向量数据库（持久化）
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)
print("向量数据库创建完成！")

# 6. 设置用户问题
user_question = "方闻馆详细时间"  # 你可以修改

# 7. 检索相似块（MMR检索）
retrieved_docs = vectorstore.max_marginal_relevance_search(user_question, k=5, fetch_k=10)
print(f"检索到的相关块（共 {len(retrieved_docs)} 个）：")
for i, doc in enumerate(retrieved_docs):
    print(f"块{i+1}: {doc.page_content[:150]}...")

# 8. 配置聊天模型
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="deepseek-ai/DeepSeek-V3",
    openai_api_key=os.getenv("siliconFlow_API_KEY"),
    openai_api_base="https://api.siliconflow.cn/v1",
    temperature=0.3
)

# 9. 定义 RAG 问答函数
def rag_answer(user_question, vectorstore, k=5):
    docs = vectorstore.similarity_search(user_question, k=k)
    context = "\n\n".join([doc.page_content for doc in docs])
    prompt = f"""你是一个校园助手。请根据以下参考资料回答用户的问题。
如果参考资料中没有相关信息，请如实说“我不知道”。

参考资料：
{context}

问题：{user_question}
回答："""
    response = llm.invoke(prompt)
    return response.content

# 10. 测试问答
answer = rag_answer(user_question, vectorstore, k=5)
print("\n最终答案：", answer)

向量数据库创建完成！
检索到的相关块（共 5 个）：
块1: 一、借阅权限
读者类型              外借册数    预约册数    预约催还过期停借册数
─────────────────────────────────────────────────────────────
在校教职工            100册       30册     ...
块2: 企编教工              30册        3册         1册
访问学者              30册        3册         1册
离退休人员            5册         3册         1册
校内其他人员          5册    ...
块3: 浙江大学图书馆使用指南
═══════════════════════════════════════════════════════

【图书馆概况】
浙江大学图书馆是浙江省规模最大、设备最齐全的高等学校图书馆之一，由紫金港校区的主馆、基础馆、农医馆、古籍馆、方闻馆和玉泉分馆、西溪分馆、华家池分馆...
块4: 【联系方式】
─────────────────────────────────
馆长信箱：infolib@zju.edu.cn
留言板：http://libweb.zju.edu.cn/55982/list.htm
海宁国际校区图书馆：0571-87572288

════════════════...
块5: 三、方闻馆详细时间
─────────────────────────────────
咨询台：周一至周日 8:30-22:30
东阅览室：周一至周日 8:30-22:30
西阅览室：周一至周日 8:30-22:30
珍本阅览室：周一至周五 08:30-12:00；13:30-17:30（周六日不开...

最终答案： 我不知道。


In [3]:

    
# 5. 创建向量数据库（持久化）
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)
print("向量数据库创建完成！")

# 6. 设置用户问题
user_question = "方闻馆详细时间"  # 你可以修改

from langchain.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever

# 创建 BM25 检索器（基于原始 chunks，需要文本列表）
bm25_retriever = BM25Retriever.from_texts([chunk.page_content for chunk in chunks])
bm25_retriever.k = 3

# 创建向量检索器
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# 组合，权重各 0.5
ensemble_retriever = EnsembleRetriever(retrievers=[bm25_retriever, vector_retriever],weights=[0.9, 0.1])

# 使用组合检索器
docs = ensemble_retriever.get_relevant_documents(user_question)
print(f"混合检索返回 {len(docs)} 个块")
for i, doc in enumerate(docs):
    print(f"块{i+1}: {doc.page_content[:150]}...")


# 7. 检索相似块


# 8. 配置聊天模型
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="deepseek-ai/DeepSeek-V3",
    openai_api_key=os.getenv("siliconFlow_API_KEY"),
    openai_api_base="https://api.siliconflow.cn/v1",
    temperature=0.3
)

# 9. 定义 RAG 问答函数
def rag_answer(user_question, retriever, k=3):
    docs = retriever.get_relevant_documents(user_question)
    context = "\n\n".join([doc.page_content for doc in docs])
    prompt = f"""你是一个校园助手。请根据以下参考资料回答用户的问题。
如果参考资料中没有相关信息，请如实说“我不知道”。

参考资料：
{context}

问题：{user_question}
回答："""
    response = llm.invoke(prompt)
    return response.content

# 10. 测试问答
answer = rag_answer(user_question, ensemble_retriever)
print("\n最终答案：", answer)

向量数据库创建完成！
混合检索返回 6 个块
块1: 【联系方式】
─────────────────────────────────
馆长信箱：infolib@zju.edu.cn
留言板：http://libweb.zju.edu.cn/55982/list.htm
海宁国际校区图书馆：0571-87572288

════════════════...
块2: 【芸悦读平台】
─────────────────────────────────
1. 借阅数上限：15本
2. 还书专窗：各校区分馆设有芸悦读还书专窗
3. 暑假期间借阅期自动延长（根据校历调整）

【文明行为规范】
─────────────────────────────────
1. 不占...
块3: 四、图书预约
─────────────────────────────────
普通流通库图书、样本库图书、后备库图书等提供外借预约和在架书预约服务。

五、逾期处理
─────────────────────────────────
如未按照应还日期归还图书，需按照每日每本0.05元的罚金缴纳罚...
块4: 一、借阅权限
读者类型              外借册数    预约册数    预约催还过期停借册数
─────────────────────────────────────────────────────────────
在校教职工            100册       30册     ...
块5: 企编教工              30册        3册         1册
访问学者              30册        3册         1册
离退休人员            5册         3册         1册
校内其他人员          5册    ...
块6: 浙江大学图书馆使用指南
═══════════════════════════════════════════════════════

【图书馆概况】
浙江大学图书馆是浙江省规模最大、设备最齐全的高等学校图书馆之一，由紫金港校区的主馆、基础馆、农医馆、古籍馆、方闻馆和玉泉分馆、西溪分馆、华家池分馆...


C:\Users\naizz\AppData\Local\Temp\ipykernel_26512\1136722862.py:26: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  docs = ensemble_retriever.get_relevant_documents(user_question)



最终答案： 我不知道。


In [7]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor

# 创建压缩器（使用你的 llm）
compressor = LLMChainExtractor.from_llm(llm)

# 基础检索器（向量或混合）
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

# 压缩检索器
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)

# 使用
docs = compression_retriever.get_relevant_documents("方闻馆开放时间")
print(f"压缩后保留 {len(docs)} 个块")

# 9. 定义 RAG 问答函数
def rag_answer(user_question, retriever, k=3):
    docss = retriever.get_relevant_documents(user_question)
    context = "\n\n".join([doc.page_content for doc in docs])
    prompt = f"""你是一个校园助手。请根据以下参考资料回答用户的问题。
如果参考资料中没有相关信息，请如实说“我不知道”。

参考资料：
{docs}

问题：{user_question}
回答："""
    response = llm.invoke(prompt)
    return response.content

# 10. 测试问答
answer = rag_answer(user_question, base_retriever)
print("\n最终答案：", answer)

压缩后保留 2 个块

最终答案： 方闻馆的详细开放时间如下：

1. 咨询台：周一至周日 8:30-22:30  
2. 东阅览室：周一至周日 8:30-22:30  
3. 西阅览室：周一至周日 8:30-22:30  
4. 珍本阅览室：周一至周五 08:30-12:00；13:30-17:30（周六日不开放）  
5. 三层密集书库：周一至周五 08:30-12:00；13:30-17:30（周六日不开放）  

如有其他问题，欢迎随时咨询！


In [ ]:
print(docs)